# Post-training — Analyse des résultats (5000 audiogrammes)

Ce notebook charge les résultats produits par `python main.py --data data/`  
Il ne réentraîne rien — il lit `results/abs/scores.csv` et les fichiers JSON sources.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.loader import load_dataset
from src.features import build_feature_matrix, STANDARD_FREQS
from src.evaluate import plot_audiogram

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

RESULTS_DIR = Path('../results/abs')
DATA_DIR    = Path('../data')
GENDER_LABEL = {1: 'Homme', 2: 'Femme'}

In [ ]:
# Chargement scores
scores_df = pd.read_csv(RESULTS_DIR / 'scores.csv', index_col=0)

# Chargement données brutes
df = load_dataset(DATA_DIR)
feature_df, _ = build_feature_matrix(df)

# Alignement index
scores_df.index = df.index
feature_df.index = df.index

# Ajout source_file dans scores pour les analyses par fichier
scores_df['source_file'] = df['source_file'].values

n_files = df['source_file'].nunique()
print(f"Dataset  : {len(df)} records — {df['patient'].nunique()} patients — {n_files} fichiers")
print(f"Scores   : {len(scores_df)} lignes")
print(f"Colonnes : {list(scores_df.columns)}")

## 1. Résumé global des flags

In [ ]:
flag_cols = ['anomaly_flag_if', 'anomaly_flag_ae', 'anomaly_flag_pca',
             'anomaly_consensus', 'nihl_flag', 'meniere_flag', 'sts_flag', 'anomaly_final']
flag_cols = [c for c in flag_cols if c in scores_df.columns]

summary = pd.DataFrame({
    'n_flaggés': [scores_df[c].fillna(0).sum() for c in flag_cols],
    '% total':   [100 * scores_df[c].fillna(0).mean() for c in flag_cols],
}, index=flag_cols).round(1)

print(summary.to_string())

# Plages épidémiologiques attendues
expected = {'nihl_flag': (15, 30), 'meniere_flag': (1, 5), 'sts_flag': (5, 15)}
print("\nCalibration vs épidémiologie attendue :")
for col, (lo, hi) in expected.items():
    if col in scores_df.columns:
        rate = 100 * scores_df[col].fillna(0).mean()
        status = '✓' if lo <= rate <= hi else '⚠'
        print(f"  {status} {col:20s} {rate:.1f}% (attendu {lo}–{hi}%)")

## 2. Analyse par fichier source

Pour choisir quel fichier labelliser en priorité.

In [ ]:
per_file = scores_df.groupby('source_file').agg(
    n_records=('anomaly_final', 'count'),
    pct_nihl=('nihl_flag', lambda x: 100 * x.fillna(0).mean()),
    pct_meniere=('meniere_flag', lambda x: 100 * x.fillna(0).mean()),
    pct_consensus_ml=('anomaly_consensus', lambda x: 100 * x.fillna(0).mean()),
    pct_final=('anomaly_final', lambda x: 100 * x.fillna(0).mean()),
    reconstruction_error_mean=('reconstruction_error', 'mean'),
).round(1)

# Ajouter stats démographiques
demo = df.groupby('source_file').agg(
    age_mean=('age_at_visit', 'mean'),
    pct_male=('gender', lambda x: 100 * (x == 1).mean()),
).round(1)

per_file = per_file.join(demo)
print(per_file.to_string())

In [ ]:
files = sorted(scores_df['source_file'].unique())
n = len(files)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# % flags par fichier
x = range(n)
width = 0.25
for i, (col, label, color) in enumerate([
    ('nihl_flag', 'NIHL', 'steelblue'),
    ('anomaly_consensus', 'Consensus ML', 'darkorange'),
    ('anomaly_final', 'Final', 'tomato'),
]):
    if col not in scores_df.columns:
        continue
    vals = [100 * scores_df[scores_df['source_file'] == f][col].fillna(0).mean() for f in files]
    axes[0].bar([xi + i * width for xi in x], vals, width=width, label=label, color=color, alpha=0.85)

axes[0].set_xticks([xi + width for xi in x])
axes[0].set_xticklabels(files, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('% flaggés')
axes[0].set_title('Taux de flags par fichier')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# Distribution âge par fichier
sns.boxplot(data=df, x='source_file', y='age_at_visit', ax=axes[1], palette='Set2')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[1].set_xlabel('')
axes[1].set_ylabel('Âge (années)')
axes[1].set_title('Distribution âge par fichier')
axes[1].grid(axis='y', alpha=0.3)

# Score reconstruction par fichier
sns.boxplot(data=scores_df, x='source_file', y='reconstruction_error', ax=axes[2], palette='Set2')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[2].set_xlabel('')
axes[2].set_ylabel('Erreur reconstruction')
axes[2].set_title('Score anomalie Autoencoder par fichier')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Distribution des scores ML

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title, color in zip(axes,
    ['anomaly_score_if', 'reconstruction_error', 'pca_reconstruction_error'],
    ['Isolation Forest', 'Autoencoder', 'PCA baseline'],
    ['steelblue', 'darkorange', 'mediumseagreen']
):
    if col not in scores_df.columns:
        continue
    vals = scores_df[col].dropna()
    ax.hist(vals, bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(vals.quantile(0.95), color='red', linestyle='--', linewidth=1.5,
               label=f'P95 = {vals.quantile(0.95):.3f}')
    ax.set_title(title)
    ax.set_xlabel('Score')
    ax.set_ylabel('Nombre de records')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Distribution des scores d\'anomalie', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Accord ML vs règles cliniques

Les cas en désaccord sont les plus intéressants à labelliser.

In [ ]:
if 'anomaly_consensus' in scores_df.columns and 'nihl_flag' in scores_df.columns:
    ml   = scores_df['anomaly_consensus'].fillna(0).astype(int)
    nihl = scores_df['nihl_flag'].fillna(0).astype(int)

    categories = {
        'Accord : tous normaux':    ((ml == 0) & (nihl == 0)).sum(),
        'Accord : tous anormaux':   ((ml == 1) & (nihl == 1)).sum(),
        'ML seul (règle rate)':     ((ml == 1) & (nihl == 0)).sum(),
        'Règle seule (ML rate)':    ((ml == 0) & (nihl == 1)).sum(),
    }

    for label, count in categories.items():
        print(f"  {label:35s}: {count:4d} ({100*count/len(ml):.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Camembert
    colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
    axes[0].pie(categories.values(), labels=categories.keys(), colors=colors,
                autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
    axes[0].set_title('Accord ML vs règle NIHL')

    # Heatmap désaccord
    cross = pd.crosstab(
        ml.map({0: 'ML: normal', 1: 'ML: anomalie'}),
        nihl.map({0: 'Règle: normal', 1: 'Règle: NIHL'})
    )
    sns.heatmap(cross, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                linewidths=0.5, cbar=False)
    axes[1].set_title('ML × Règle NIHL')

    plt.tight_layout()
    plt.show()

## 5. Profil audiométrique moyen — NIHL flaggé vs non flaggé

In [ ]:
if 'nihl_flag' in scores_df.columns:
    nihl_mask = scores_df['nihl_flag'].fillna(0).astype(bool)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, ear, label in zip(axes, ['L', 'R'], ['Oreille gauche (OG)', 'Oreille droite (OD)']):
        cols = [f'{ear}_{f}' for f in STANDARD_FREQS if f'{ear}_{f}' in feature_df.columns]
        freqs = [f for f in STANDARD_FREQS if f'{ear}_{f}' in feature_df.columns]
        if not cols:
            continue

        for mask, color, name in [
            (nihl_mask,  'tomato',    f'NIHL flaggé (n={nihl_mask.sum()})'),
            (~nihl_mask, 'steelblue', f'Non flaggé (n={(~nihl_mask).sum()})'),
        ]:
            subset = feature_df.loc[mask, cols]
            mean = subset.mean()
            std  = subset.std()
            ax.plot(freqs, mean.values, color=color, linewidth=2.5, label=name)
            ax.fill_between(freqs, mean - std, mean + std, alpha=0.12, color=color)

        ax.axhline(25, color='gray', linestyle=':', alpha=0.5, label='Norme 25 dB')
        ax.set_xscale('log')
        ax.set_xticks(freqs)
        ax.set_xticklabels([f'{f}' if f < 1000 else f'{f//1000}k' for f in freqs])
        ax.invert_yaxis()
        ax.set_ylim(90, -15)
        ax.set_xlabel('Fréquence (Hz)')
        ax.set_ylabel('Seuil auditif moyen (dB HL)')
        ax.set_title(label)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.25)

    plt.suptitle('Profil moyen NIHL flaggé vs non flaggé (±1σ)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Zone grise — cas prioritaires pour labellisation

Les cas avec un score entre P50 et P90 sont les plus incertains pour le modèle.  
C'est là que la labellisation apporte le plus de valeur.

In [ ]:
if 'reconstruction_error' in scores_df.columns:
    score = scores_df['reconstruction_error']
    p50, p90 = score.quantile(0.50), score.quantile(0.90)

    grey_zone = (score >= p50) & (score <= p90)
    high_conf_anomaly = score > p90
    high_conf_normal  = score < p50

    print(f"Cas haute confiance normaux  (score < P50) : {high_conf_normal.sum():4d} ({100*high_conf_normal.mean():.0f}%)")
    print(f"Zone grise           (P50–P90)            : {grey_zone.sum():4d} ({100*grey_zone.mean():.0f}%)")
    print(f"Cas haute confiance anomalies (score > P90): {high_conf_anomaly.sum():4d} ({100*high_conf_anomaly.mean():.0f}%)")

    # Répartition zone grise par fichier
    grey_by_file = scores_df[grey_zone].groupby('source_file').size()
    total_by_file = scores_df.groupby('source_file').size()
    pct_grey = (100 * grey_by_file / total_by_file).round(1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Distribution score colorée par zone
    axes[0].hist(score[high_conf_normal],  bins=40, color='steelblue',  alpha=0.7, label=f'Normal confiant (< P50)')
    axes[0].hist(score[grey_zone],          bins=40, color='orange',      alpha=0.7, label=f'Zone grise (P50–P90)')
    axes[0].hist(score[high_conf_anomaly],  bins=40, color='tomato',      alpha=0.7, label=f'Anomalie confiante (> P90)')
    axes[0].axvline(p50, color='orange', linestyle='--', linewidth=1.5)
    axes[0].axvline(p90, color='red',    linestyle='--', linewidth=1.5)
    axes[0].set_xlabel('Erreur de reconstruction')
    axes[0].set_ylabel('Nombre de records')
    axes[0].set_title('Zones de confiance du modèle')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    # % zone grise par fichier
    pct_grey.plot(kind='bar', ax=axes[1], color='orange', edgecolor='white', alpha=0.85)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
    axes[1].set_ylabel('% de records en zone grise')
    axes[1].set_title('Zone grise par fichier source')
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

## 7. Top anomalies — visualisation audiogrammes

In [ ]:
N_SHOW = 6

if 'reconstruction_error' in scores_df.columns:
    top_idx = scores_df['reconstruction_error'].nlargest(N_SHOW).index
    fig, axes = plt.subplots(2, N_SHOW // 2, figsize=(14, 8))
    axes = axes.flatten()

    for i, idx in enumerate(top_idx):
        row   = df.loc[idx]
        score = scores_df.loc[idx, 'reconstruction_error']
        nihl  = int(scores_df.loc[idx, 'nihl_flag']) if 'nihl_flag' in scores_df.columns else '?'
        src   = scores_df.loc[idx, 'source_file'] if 'source_file' in scores_df.columns else ''
        title = f"#{idx} — score={score:.3f}\nNIHL={nihl} | {src}"
        plot_audiogram(row['dots_left'], row['dots_right'], title=title, ax=axes[i])

    plt.suptitle(f'Top {N_SHOW} anomalies (erreur de reconstruction)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. Export — cas à labelliser

Génère le CSV stratifié à envoyer pour labellisation audiologiste :
- ~300 cas flaggés NIHL (rule-positive)
- ~300 cas en zone grise (modèle incertain)
- Ces deux groupes maximisent la valeur de la labellisation

In [ ]:
N_PER_GROUP = 300

if 'nihl_flag' in scores_df.columns and 'reconstruction_error' in scores_df.columns:
    # Groupe 1 : cas flaggés NIHL par les règles
    nihl_positive = scores_df[scores_df['nihl_flag'] == 1].sample(
        n=min(N_PER_GROUP, (scores_df['nihl_flag'] == 1).sum()), random_state=42
    )

    # Groupe 2 : zone grise (P50–P90), non déjà dans groupe 1
    score = scores_df['reconstruction_error']
    p50, p90 = score.quantile(0.50), score.quantile(0.90)
    grey_pool = scores_df[
        (score >= p50) & (score <= p90) & (scores_df['nihl_flag'] != 1)
    ]
    grey_sample = grey_pool.sample(
        n=min(N_PER_GROUP, len(grey_pool)), random_state=42
    )

    to_label = pd.concat([nihl_positive, grey_sample]).drop_duplicates()
    to_label['labeling_group'] = [
        'nihl_positive' if idx in nihl_positive.index else 'grey_zone'
        for idx in to_label.index
    ]

    # Ajouter infos patient pour l'audiologiste
    export = to_label[['source_file', 'labeling_group', 'nihl_flag',
                        'reconstruction_error', 'anomaly_consensus']].copy()
    export['patient']        = df.loc[export.index, 'patient'].values
    export['visit_date']     = df.loc[export.index, 'visit_date'].values
    export['visit_category'] = df.loc[export.index, 'visit_category'].values
    export['age_at_visit']   = df.loc[export.index, 'age_at_visit'].values
    export['gender']         = df.loc[export.index, 'gender'].values
    export['nihl_label']     = ''   # à remplir par l'audiologiste
    export['confidence']     = ''   # haut / moyen / faible
    export['comment']        = ''

    out_path = Path('../labels/batch_01_to_label.csv')
    out_path.parent.mkdir(exist_ok=True)
    export.to_csv(out_path)

    print(f"Export : {len(export)} cas → {out_path}")
    print(f"  Groupe NIHL positif : {(to_label['labeling_group'] == 'nihl_positive').sum()}")
    print(f"  Groupe zone grise   : {(to_label['labeling_group'] == 'grey_zone').sum()}")
    print(f"\nRépartition par fichier :")
    print(export.groupby(['source_file', 'labeling_group']).size().to_string())